# Text → gloss datasets: ASLG-PC12 splits, WLASL/GISLR lexicons, team30

**What this notebook does.** Pipeline stage (dataset), TODO §13 Phase 2. It prepares
everything `aslg.1.models.text2gloss.ipynb` scores against:

| artifact | path | what |
|---|---|---|
| ASLG-PC12 raw | `data/raw/aslg_pc12/data/train-00000-of-00001.parquet` | 87,710 English/gloss pairs (HF `achrafothman/aslg_pc12`) |
| fixed splits | `data/cache/aslg_pc12/splits/v1/{train,val,test}.parquet` + `split_info.json` | 80/10/10 **by unique English text**, seed from the config |
| WLASL index | `data/raw/wlasl/WLASL_v0.3.json` | 2,000 glosses + instance trim windows (no videos) |
| team30 | `sb.synthesize.evalsets` (committed) | the team's 30 sentences, their T5's recorded output, **draft** references |

**Resumable:** every step skips work whose output exists.

**Design decisions vs the TODO §13 plan**
- The split is by **unique text**: 6,587 English texts repeat in ASLG-PC12, so a row split
  would put the same sentence in train and test.
- **NCSLGR** has no scriptable download (BU licence form). §2 only checks for a manual export.
- ASLG-PC12's glosses are **synthetic** (`DESC-`/`X-` markers, `BE` kept). §4 measures how
  far that convention is from the team30 references, which is why aslg.1 scores it with
  a normalizer and reads it as "reproduce that rule system", not "real ASL".

## Setup

In [1]:
# ============================================================
# Imports, config, paths
# ============================================================
import json

import pandas as pd

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, ROOT_DIR
from sb.synthesize import data as D
from sb.synthesize import evalsets, lexicon, metrics
from sb.synthesize.gloss import GlossEngine, hybrid, t5

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
CONFIG = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
T5_DIR = ROOT_DIR / CONFIG["t5"]["model_dir"]
OUT = CACHE_DIR / "synthesis" / "text2gloss"
OUT.mkdir(parents=True, exist_ok=True)

print(f"ASLG-PC12 raw : {D.ASLG_RAW}")
print(f"splits        : {D.ASLG_CACHE / 'splits' / CONFIG['splits']['version']}")
print(f"WLASL index   : {lexicon.WLASL_JSON}")
print(f"T5 checkpoint : {T5_DIR} -> {'FOUND' if t5.available(T5_DIR) else 'missing (hybrids fall back to rules)'}")
print(f"results       : {OUT}")

ASLG-PC12 raw : C:\Users\user2\repository\data\raw\aslg_pc12
splits        : C:\Users\user2\repository\data\cache\aslg_pc12\splits\v1
WLASL index   : C:\Users\user2\repository\data\raw\wlasl\WLASL_v0.3.json
T5 checkpoint : C:\Users\user2\repository\data\external\t5-text2gloss\thesis_hybrid_dataset1\model -> missing (hybrids fall back to rules)
results       : C:\Users\user2\repository\data\cache\synthesis\text2gloss


## 1. ASLG-PC12: download and fixed splits

Downloads ~7.5 MB once, dedups by English text, and writes the seeded splits.

In [2]:
# ============================================================
# ASLG-PC12 -> data/cache/aslg_pc12/splits/<version>/
# ============================================================
sp = CONFIG["splits"]
splits = D.aslg_splits(sp["fractions"], sp["seed"], sp["version"])
info = D.split_info(sp["version"])
print(json.dumps(info, indent=1))
leak = set(splits["train"].text) & set(splits["test"].text)
print(f"train/test text overlap: {len(leak)}")
splits["test"].sample(8, random_state=0)

{
 "source": "hf:achrafothman/aslg_pc12/data/train-00000-of-00001.parquet",
 "seed": 42,
 "fractions": {
  "train": 0.8,
  "val": 0.1,
  "test": 0.1
 },
 "unique_texts": 81088,
 "sizes": {
  "train": 64870,
  "val": 8109,
  "test": 8109
 }
}
train/test text overlap: 0


,gloss,text
4581,X-I WOULD LIKE TO COMPLIMENT SWEDISH PRESIDENC...,i would like to compliment the swedish preside...
1845,WHAT DESC-RIGHT TO ABORTION HAVE TO DO WITH CO...,what a right to abortion has to do with combat...
6408,THIS PROJECT CAUSE DESC-STORMY DEBATE AT DESC-...,this project caused stormy debate at almost al...
6601,THOSE VOTE REFLECT DESC-NOTICEABLE LACK COMMUN...,those votes reflected a noticeable lack of com...
257,IDEA THAT ECB HAVE DO DESC-GOOD JOB BE DESC-EX...,the idea that the ecb has done a good job is e...
1917,DESC-TOGER WITH HARMONISATION SAFETY STANDARD ...,together with the harmonisation of safety stan...
2235,"THANK X-YOU , SARAH , FOR X-YOU DESC-CLOSE COO...","thank you , sarah , for your close cooperation ."
451,X-IT BE THIS WHICH DRIVE REPORT X-POSS DEMAND ...,it is this which drives the report's demand fo...


## 2. NCSLGR (manual)

Real, linguistically annotated signing (BU ASLLRP). Request it through the licence form,
export `english,gloss` pairs to `data/raw/ncslgr/ncslgr.csv`, and re-run. Until then aslg.1
skips it.

In [3]:
# ============================================================
# NCSLGR presence check
# ============================================================
nc = D.ncslgr()
print("NCSLGR:", "missing -> " + str(D.NCSLGR_CSV) if nc is None else f"{len(nc)} pairs")

NCSLGR: missing -> C:\Users\user2\repository\data\raw\ncslgr\ncslgr.csv


## 3. Lexicons: WLASL index and GISLR's 250 signs

Used for the coverage metric (share of gloss units that are real signs vs fingerspelled)
and, later, for renderer A (trimmed WLASL clips). Videos are not downloaded; see
`sb.synthesize.lexicon` for where to put any you have.

In [4]:
# ============================================================
# WLASL index + GISLR label list
# ============================================================
lexicon.ensure_wlasl_index()
lex = lexicon.Lexicon()
n_inst = sum(len(v) for v in lex.wlasl.values())
multi = sorted(g for g in lex.wlasl if " " in g)
videos = len(list(lex.videos_dir.glob("*.mp4"))) if lex.videos_dir.is_dir() else 0
print(f"WLASL: {len(lex.wlasl)} glosses, {n_inst} instances, {len(multi)} multi-word (e.g. {multi[:6]})")
print(f"WLASL videos on disk: {videos}")
print(f"GISLR: {len(lex.gislr)} signs; in WLASL too: {len(lex.gislr & set(lex.wlasl))}")

WLASL: 2000 glosses, 21083 instances, 45 multi-word (e.g. ['a lot', 'all day', 'cochlear implant', 'come here', 'common sense', 'dining room'])
WLASL videos on disk: 0
GISLR: 250 signs; in WLASL too: 201


## 4. team30, and how far ASLG-PC12's convention is from it

team30's references are **drafts** (written by Claude, unreviewed; see
`evalsets/team30.v1.json` → `reference_status`). This cell also pins the frozen
engine: `rules_v1` must reproduce the rule-only outputs recorded in the audit
(`docs/reports/speech-to-sign-audit.md` §2). A mismatch means the engine drifted from
the team's `rule_engine.py`, which would silently degrade T5.

In [5]:
# ============================================================
# team30 + rules_v1 pin + convention gap
# ============================================================
from sb.synthesize.gloss import rules_v1

t30 = evalsets.load("team30", CONFIG["eval"]["team30_version"])
AUDIT_RULE_ONLY = {  # report §2, "Rule engine only" column
    1: "I GO HOME NOW", 2: "SHE LIKE COFFEE", 4: "WE BUY NEW CAR YESTERDAY", 11: "I NOT LIKE COLD WEATHER",
    12: "SHE NOT GO WORK YESTERDAY", 18: "YOU LEAVE EARLY WHY", 19: "YOU CALL YOUR MOTHER",
    20: "YOU HELP I TOMORROW", 21: "GIVE I GLASS WATER", 25: "TELL SHE I FUTURE COME LATER",
    26: "IT RAIN TOMORROW I FUTURE STAY HOME", 29: "YOU LEAVE TURN LIGHT",
}
bad = {i: (rules_v1.convert(t30.english[i - 1]), g) for i, g in AUDIT_RULE_ONLY.items()
       if rules_v1.convert(t30.english[i - 1]) != g}
assert not bad, f"rules_v1 drifted from the team's engine: {bad}"
print(f"rules_v1 pin: {len(AUDIT_RULE_ONLY)}/{len(AUDIT_RULE_ONLY)} audit outputs reproduced")


def style(glosses):
    toks = [t for g in glosses for t in str(g).split()]
    n = max(len(toks), 1)
    return {"tokens/sent": len(toks) / len(glosses),
            "DESC-/X- marked": sum(t.startswith(("DESC-", "X-")) for t in toks) / n,
            "BE": sum(t == "BE" for t in toks) / n,
            "ME": sum(t in {"ME", "X-ME"} for t in toks) / n,
            "I": sum(t in {"I", "X-I"} for t in toks) / n,
            "punct": sum(not any(c.isalnum() for c in t) for t in toks) / n}


pd.DataFrame({"ASLG-PC12 test": style(splits["test"].gloss), "team30 draft refs": style(t30.reference_draft),
              "team T5 (recorded)": style(t30.hybrid_recorded)}).round(3)

rules_v1 pin: 12/12 audit outputs reproduced


,ASLG-PC12 test,team30 draft refs,team T5 (recorded)
tokens/sent,12.224,5.067,5.133
DESC-/X- marked,0.206,0.000,0.000
BE,0.052,0.000,0.000
ME,0.000,0.092,0.084
I,0.015,0.000,0.000
punct,0.116,0.000,0.000
